In [1]:
import networkx as nx;
import pandas as pd;
import gurobipy as gp;
from gurobipy import GRB;
import csv;
import sys;
import numpy

In [19]:
# networkCSV = 'TestInstances/CSV_TestInstances/N100/' + 'N100_22.csv';
# N = 1000; #sample size
# budget = 5;
# numpy.random.seed(2024);

networkCSV = './NewCSVFeb24/N10_1.csv'
N = 10
budget = 2
numpy.random.seed(2024)


In [20]:
# Reading network file
with open(networkCSV, newline='') as f:
    reader = csv.reader(f);
    row1 = next(reader);
    nbArcs = int(row1[0]);
    row2 = next(reader);
    s = int(row2[0]);
    row3 = next(reader);
    t = int(row3[0]);
    
    G = nx.DiGraph();
    data = pd.read_csv(networkCSV, skiprows=4, header=None, delim_whitespace=True);
    n_edge = len(data.index);

    for i in range(n_edge): 
        G.add_edge(data.iat[i,0], data.iat[i,1], costLB = data.iat[i,2], 
                costUB = data.iat[i,3], interEffect = data.iat[i,4], tempCost = 0);

/var/folders/cy/c941bw5943jb9bfdpqrp1g180000gn/T/ipykernel_94288/2165234358.py:12: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(networkCSV, skiprows=4, header=None, delim_whitespace=True);


In [21]:
'''
print("nbArcs = ", nbArcs);
print("s = ", s);
print("t = ", t);
print("G.nodes = ", G.nodes)
print("G.edges = ", G.edges)
for e in G.edges:
    print(e)
    print(G.edges[e])
'''

'\nprint("nbArcs = ", nbArcs);\nprint("s = ", s);\nprint("t = ", t);\nprint("G.nodes = ", G.nodes)\nprint("G.edges = ", G.edges)\nfor e in G.edges:\n    print(e)\n    print(G.edges[e])\n'

In [22]:
scens = [];
for k in range(N):
    scen = {};
    for e in G.edges:
        scen[e] = numpy.random.uniform(G.edges[e]['costLB'],G.edges[e]['costUB']);
    scens.append(scen);

In [23]:
print("scens[0] = ", scens[0])

scens[0] =  {(1, 2): 17.760290377907957, (1, 4): 132.0, (1, 8): 2.0, (2, 8): 65.0, (2, 10): 236.0, (4, 2): 42.0, (4, 5): 7.0, (4, 10): 197.0, (8, 3): 172.0, (8, 4): 67.4500325542668, (8, 6): 65.0, (8, 7): 34.0, (8, 9): 31.0, (3, 2): 20.0, (3, 4): 6.0, (3, 9): 149.0, (9, 2): 137.0, (9, 3): 200.0, (9, 4): 245.0, (9, 6): 67.50366147163393, (9, 7): 87.0, (9, 10): 16.0, (5, 3): 38.0, (5, 10): 99.0, (7, 2): 167.41261938313622, (7, 3): 93.0, (7, 4): 33.0, (7, 6): 40.0, (7, 8): 42.0, (7, 9): 1.0, (7, 10): 89.0}


In [24]:
# Callback - use lazy constraints
def lazy(model, where):
    if where == GRB.Callback.MIPSOL:
        xvals = model.cbGetSolution(model._x)
        thetavals = model.cbGetSolution(model._theta);
        for k in range(len(model._scens)):
            # multi-cut version
            # update edge cost per scenario
            for e in model._G.edges:
                if xvals[e] > 1e-5:
                    model._G.edges[e]['tempCost'] = model._scens[k][e] + model._G.edges[e]['interEffect'];
                else:
                    model._G.edges[e]['tempCost'] = model._scens[k][e];
            # obtain the shortest path and its length
            spValue = nx.shortest_path_length(model._G, source=model._s, target=model._t, weight='tempCost', method='dijkstra')
            if spValue < thetavals[k]-(1e-5):
                # add lazy constraints
                spPath = nx.shortest_path(model._G, source=model._s, target=model._t, weight='tempCost', method='dijkstra')
                constrCoefList = [1];
                constrVarList = [model._theta[k]];
                rhs = 0
                for i in range(len(spPath)-1):
                    rhs += model._scens[k][(spPath[i],spPath[i+1])];
                    constrCoefList.append(-model._G.edges[(spPath[i],spPath[i+1])]['interEffect']);
                    constrVarList.append(model._x[(spPath[i],spPath[i+1])]);
                expr = gp.LinExpr();
                expr.addTerms(constrCoefList, constrVarList);
                model.cbLazy(expr <= rhs);

In [25]:
master = gp.Model()

# Create variables
x = {};
for e in G.edges:
    x[e] = master.addVar(obj=0, vtype=GRB.BINARY);
    
theta = {};
for k in range(N):
    theta[k] = master.addVar(obj=1.0/N, vtype=GRB.CONTINUOUS, lb = 0, ub = 1e7);

# Add interdiction budget constraint
master.addConstr(gp.quicksum(x[e] for e in G.edges) <= budget);

master._x = x
master._theta = theta
master._G = G
master._s = s
master._t = t
master._scens = scens

In [26]:
master.modelSense = GRB.MAXIMIZE
master.Params.LazyConstraints = 1
master.optimize(lazy)

xvals = master.getAttr('X', x)

print('')
print('Optimal objval: %g' % master.ObjVal)
print('')
print('Optimal xval = ')
for e in G.edges:
    if xvals[e] > 1e-5:
        print(e);
        print(" ")

Set parameter LazyConstraints to value 1
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (mac64[arm])

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 1 rows, 41 columns and 31 nonzeros
Model fingerprint: 0x63b752fd
Variable types: 10 continuous, 31 integer (31 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e-01, 1e-01]
  Bounds range     [1e+00, 1e+07]
  RHS range        [2e+00, 2e+00]
Presolve time: 0.00s
Presolved: 1 rows, 41 columns, 31 nonzeros
Variable types: 10 continuous, 31 integer (31 binary)

Root relaxation: objective 7.600000e+01, 11 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0      76.0000000   76.00000  0.00%     -    0s

Cutting planes:
  Lazy constraints: 10

Explored 1 nodes 